# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we display the available record sets (`@id`s), list the fields in each, and enumerate their corresponding `@id`s.

In [ ]:
# List all available Record Sets
record_sets = []
for record_set in metadata.record_sets:
    print(f"RecordSet: @id={record_set.id}, name={getattr(record_set, 'name', None)}")
    record_sets.append(record_set.id)
    fields = getattr(record_set, 'fields', [])
    if fields:
        print("  Fields/columns in this RecordSet:")
        for field in fields:
            print(f"    Field @id={field.id}, name={getattr(field, 'name', None)}, type={getattr(field, 'data_type', None)}")
    print('-' * 60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set and load as pandas DataFrames
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only include non-empty record sets
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records – columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        print("  No records available in this set.")
    print('-' * 40)
# Select the main record set (the one with the most records) as default for further analysis
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"Selected main record_set for EDA: {main_record_set_id}")
    print("First 5 records:")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations such as removing outliers, transforming data distributions, and grouping data by key attributes.

**Note:** All field/column references are made using their `@id`.

In [ ]:
import numpy as np

# Choose a numeric field (by @id) for analysis, e.g., GAD-7 total score
# You can change this to the proper field from the overview (see record set fields printed above)
# Example: numeric_field_id = "gad7_total_score" (replace with the actual id if needed)

numeric_field_id = None  # Replace with the correct @id from fields above (e.g., 'gad7_total_score')
if main_record_set_id:
    candidates = [col for col in dataframes[main_record_set_id].columns if 'score' in col.lower() or 'age' in col.lower()]
    print(f"Numeric field candidates: {candidates}")
    # Try to auto-select, else set manually
    if candidates:
        numeric_field_id = candidates[0]
        print(f"Using field for numeric analysis: {numeric_field_id}")
    else:
        print("No obvious numeric field found – set `numeric_field_id` manually from column list.")
    # Only proceed if a valid numeric field
    if numeric_field_id:
        # Threshold for filtering
        threshold = dataframes[main_record_set_id][numeric_field_id].mean() if np.issubdtype(dataframes[main_record_set_id][numeric_field_id].dtype, np.number) else 10
        filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-6)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field such as 'gender', 'village', or 'level_of_education'
        possible_group_fields = [col for col in dataframes[main_record_set_id].columns if any(cat in col.lower() for cat in [
            'gender', 'village', 'education', 'marital'])]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found in columns:", dataframes[main_record_set_id].columns.tolist())
    else:
        print('No numeric field found for EDA. Please check the Record Set overview.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field found, make a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset includes survey responses from Kilifi County, with demographic and mental health measures such as GAD-7, PHQ-9, PSQ.
- We demonstrated data extraction from Croissant schema, identified record sets and fields via their `@id`, and performed basic EDA and visualizations.
- You may continue to refine analyses by referencing fields via their `@id` and exploring additional relationships in the data.